In [1]:
import numpy as np
import pandas as pd

import scanpy as sc
import scvelo as scv

import matplotlib.pyplot as plt
import seaborn as sns

import myutils

import random
import os

myutils.set_figure_params()
sc.settings.verbosity = 3

In [4]:
adata = sc.read_h5ad("../1_preprocess_integration/1.3.final_all_for_open_access.h5ad")
adata

AnnData object with n_obs × n_vars = 166418 × 25998
    obs: 'cell_id', 'sample', 'n_genes', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_counts', 'condition', 'leiden', 'CellType', 'condition_final'
    var: 'n_cells', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'CellType_colors', 'batch_colors', 'condition_colors', 'hvg', 'leiden', 'leiden_colors', 'log1p', 'neighbors', 'pca', 'sample_colors', 'umap'
    obsm: 'X_pca', 'X_pca_harmony', 'X_umap'
    varm: 'PCs'
    obsp: 'connectivities', 'distances'

In [5]:
KC_meta = sc.read_h5ad("../2_keratinocyte/2.1.keratinocyte_final.h5ad").obs
KC_meta.head(5)

,cell_id,sample,n_genes,n_genes_by_counts,total_counts,total_counts_mt,pct_counts_mt,n_counts,condition,leiden,CellType,condition_final,leiden1.0,leiden0.5,CellType2
HS1_0_CELL520_N2,HS1_0_CELL520_N2,HS1,3555,3555,17932.0,634.0,3.535579,17932.0,Scar/keloid,0,Keratinocyte,Hypertrophic Scar,1,0,Spi-1
HS1_0_CELL7176_N1,HS1_0_CELL7176_N1,HS1,1277,1277,6158.0,235.0,3.816174,6158.0,Scar/keloid,0,Keratinocyte,Hypertrophic Scar,3,1,Spi-EMT
HS1_0_CELL2387_N2,HS1_0_CELL2387_N2,HS1,2055,2055,10163.0,388.0,3.817770,10163.0,Scar/keloid,0,Keratinocyte,Hypertrophic Scar,1,0,Spi-1
HS1_0_CELL676_N2,HS1_0_CELL676_N2,HS1,3194,3194,18461.0,355.0,1.922973,18461.0,Scar/keloid,0,Keratinocyte,Hypertrophic Scar,9,6,Spi-2
HS1_0_CELL3719_N1,HS1_0_CELL3719_N1,HS1,2479,2479,12169.0,629.0,5.168871,12169.0,Scar/keloid,4,Keratinocyte,Hypertrophic Scar,0,2,Bas


In [6]:
FB_meta = sc.read_h5ad("../3_fibroblasts/3.1.fibroblast_final.h5ad").obs
FB_meta.head(5)

,cell_id,sample,n_genes,n_genes_by_counts,total_counts,total_counts_mt,pct_counts_mt,n_counts,condition,leiden,CellType,condition_final,leiden1.0,CellType2
HS1_0_CELL5242_N1,HS1_0_CELL5242_N1,HS1,1162,1162,3954.0,147.0,3.717754,3954.0,Scar/keloid,2,Fibroblast,Hypertrophic Scar,0,Fibroblast (non-myo)
HS1_0_CELL9007_N1,HS1_0_CELL9007_N1,HS1,3770,3770,18157.0,436.0,2.401278,18157.0,Scar/keloid,2,Fibroblast,Hypertrophic Scar,5,Myofibroblast
HS1_0_CELL4107_N1,HS1_0_CELL4107_N1,HS1,825,825,2153.0,65.0,3.019043,2153.0,Scar/keloid,2,Fibroblast,Hypertrophic Scar,12,Fibroblast (non-myo)
HS1_0_CELL3221_N1,HS1_0_CELL3221_N1,HS1,3462,3462,13849.0,660.0,4.765687,13849.0,Scar/keloid,2,Fibroblast,Hypertrophic Scar,4,Myofibroblast
HS1_0_CELL134_N1,HS1_0_CELL134_N1,HS1,2186,2186,7660.0,278.0,3.629243,7660.0,Scar/keloid,2,Fibroblast,Hypertrophic Scar,11,Fibroblast (non-myo)


In [7]:
tmp = adata.obs['CellType'].to_numpy()
adata.obs['CellType_cellphone'] = tmp
adata.obs.loc[KC_meta['cell_id'],"CellType_cellphone"] = KC_meta['CellType2']
adata.obs.loc[FB_meta['cell_id'],"CellType_cellphone"] = FB_meta['CellType2']
adata = adata[~adata.obs['CellType_cellphone'].isin(["Fibroblast","Keratinocyte"])]
adata.obs['CellType_cellphone'].unique()

array(['Pericyte', 'T Cell', 'Fibroblast (non-myo)', 'Myofibroblast',
       'Endothelial Cell', 'Mast Cell', 'Neuron', 'Spi-1', 'Spi-EMT',
       'Spi-2', 'Bas', 'Myeloid Cell', 'Bas-pro', 'Sweat Gland Cell',
       'Granu', 'Melanocyte', 'Bas-EMT'], dtype=object)

In [9]:
os.mkdir("5.1.CellPhoneDB")

In [10]:
metadata = pd.DataFrame(adata.obs_names)
metadata['cell_type'] = adata.obs['CellType_cellphone'].tolist()
metadata.columns = ['sample_barcode',"cell_type"]
metadata.to_csv("5.1.CellPhoneDB/metadata.tsv",sep="\t",index=None)
metadata

,sample_barcode,cell_type
0,HS1_0_CELL773_N2,Pericyte
1,HS1_0_CELL2512_N1,T Cell
2,HS1_0_CELL5242_N1,Fibroblast (non-myo)
3,HS1_0_CELL9007_N1,Myofibroblast
4,HS1_0_CELL4359_N1,Endothelial Cell
...,...,...
159646,Normal4_1_CELL5174_N1,Sweat Gland Cell
159647,Normal4_1_CELL10815_N1,Bas
159648,Normal4_1_CELL3801_N1,Spi-EMT
159649,Normal4_1_CELL5675_N1,Bas


In [12]:
adata.write_h5ad("5.1.CellPhoneDB/normalised_log_counts.h5ad")

In [13]:
adata.obs['condition']

HS1_0_CELL773_N2          Scar/keloid
HS1_0_CELL2512_N1         Scar/keloid
HS1_0_CELL5242_N1         Scar/keloid
HS1_0_CELL9007_N1         Scar/keloid
HS1_0_CELL4359_N1         Scar/keloid
                             ...     
Normal4_1_CELL5174_N1              NS
Normal4_1_CELL10815_N1             NS
Normal4_1_CELL3801_N1              NS
Normal4_1_CELL5675_N1              NS
Normal4_1_CELL6177_N1              NS
Name: condition, Length: 159651, dtype: category
Categories (2, object): ['NS', 'Scar/keloid']

In [14]:
tmp = [i[0]+"_"+i[1] for i in zip(adata.obs['CellType_cellphone'],adata.obs['condition'])]
adata.obs['CellType_cellphone2'] = tmp
adata.obs['CellType_cellphone2'].unique()

array(['Pericyte_Scar/keloid', 'T Cell_Scar/keloid',
       'Fibroblast (non-myo)_Scar/keloid', 'Myofibroblast_Scar/keloid',
       'Endothelial Cell_Scar/keloid', 'Mast Cell_Scar/keloid',
       'Neuron_Scar/keloid', 'Spi-1_Scar/keloid', 'Spi-EMT_Scar/keloid',
       'Spi-2_Scar/keloid', 'Bas_Scar/keloid', 'Myeloid Cell_Scar/keloid',
       'Bas-pro_Scar/keloid', 'Sweat Gland Cell_Scar/keloid',
       'Granu_Scar/keloid', 'Melanocyte_Scar/keloid',
       'Bas-EMT_Scar/keloid', 'Endothelial Cell_NS', 'Bas_NS', 'Granu_NS',
       'Fibroblast (non-myo)_NS', 'Pericyte_NS', 'Myeloid Cell_NS',
       'Sweat Gland Cell_NS', 'Spi-EMT_NS', 'Spi-1_NS', 'Bas-pro_NS',
       'Melanocyte_NS', 'T Cell_NS', 'Neuron_NS', 'Bas-EMT_NS',
       'Mast Cell_NS', 'Spi-2_NS', 'Myofibroblast_NS'], dtype=object)

In [17]:
metadata = pd.DataFrame(adata.obs_names)
metadata['cell_type'] = adata.obs['CellType_cellphone2'].tolist()
metadata.columns = ['sample_barcode',"cell_type"]
metadata.to_csv("5.1.CellPhoneDB/metadata_by_condition.tsv",sep="\t",index=None)
metadata

,sample_barcode,cell_type
0,HS1_0_CELL773_N2,Pericyte_Scar/keloid
1,HS1_0_CELL2512_N1,T Cell_Scar/keloid
2,HS1_0_CELL5242_N1,Fibroblast (non-myo)_Scar/keloid
3,HS1_0_CELL9007_N1,Myofibroblast_Scar/keloid
4,HS1_0_CELL4359_N1,Endothelial Cell_Scar/keloid
...,...,...
159646,Normal4_1_CELL5174_N1,Sweat Gland Cell_NS
159647,Normal4_1_CELL10815_N1,Bas_NS
159648,Normal4_1_CELL3801_N1,Spi-EMT_NS
159649,Normal4_1_CELL5675_N1,Bas_NS


In [18]:
adata.write_h5ad("5.1.CellPhoneDB/normalised_log_counts_by_condition.h5ad")

In [19]:
!conda list

# packages in environment at C:\Users\dell\.conda\envs\scanpy:
#
# Name                    Version                   Build  Channel
absl-py                   2.3.0                    pypi_0    pypi
adjusttext                1.3.0                    pypi_0    pypi
aiobotocore               2.5.4                    pypi_0    pypi
aiohappyeyeballs          2.6.1                    pypi_0    pypi
aiohttp                   3.12.4                   pypi_0    pypi
aioitertools              0.12.0                   pypi_0    pypi
aiosignal                 1.3.2                    pypi_0    pypi
anndata                   0.10.8                   pypi_0    pypi
anyio                     4.8.0                    pypi_0    pypi
argon2-cffi               23.1.0                   pypi_0    pypi
argon2-cffi-bindings      21.2.0                   pypi_0    pypi
array-api-compat          1.10.0                   pypi_0    pypi
arrow                     1.3.0                    pypi_0    pypi
asciitree 